<a href="https://colab.research.google.com/github/rodrigopozza/QUANTUM_QISKIT/blob/main/extrairTCEzip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import base64
import json
import time
import requests
import zipfile  # Biblioteca nativa para lidar com arquivos ZIP
from bs4 import BeautifulSoup

# 1. Configurações fixas para Paranavaí em 2026
CODIGO_MUNICIPIO = "18402"
NOME_MUNICIPIO = "PARANAVAÍ"
ANO = "2026"

# Pasta final onde os XMLs extraídos serão salvos
PASTA_DESTINO = f"./downloads_tce_pr/{NOME_MUNICIPIO}/{ANO}"
os.makedirs(PASTA_DESTINO, exist_ok=True)

def gerar_url_consulta():
    """Gera a URL em Base64 exigida pelo TCE-PR"""
    dados_busca = {
        "cdMunicipio": CODIGO_MUNICIPIO,
        "nrAno": ANO,
        "municipio": NOME_MUNICIPIO
    }
    json_string = json.dumps(dados_busca, ensure_ascii=False)
    base64_string = base64.b64encode(json_string.encode('utf-8')).decode('utf-8')
    return f"https://pit.tce.pr.gov.br/Dados/DadosConsulta/Consulta/?f={base64_string}"

def baixar_e_extrair_zip():
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    })

    print(f"[+] Acessando o painel do TCE-PR para: {NOME_MUNICIPIO} ({ANO})...")
    url_consulta = gerar_url_consulta()

    try:
        resposta = session.get(url_consulta, timeout=15)
        if resposta.status_code != 200:
            print(f"[-] Erro ao acessar a página. Status: {resposta.status_code}")
            return

        soup = BeautifulSoup(resposta.text, 'html.parser')

        # Mapeia links de arquivos XML ou ZIP
        links_arquivos = []
        for tag_a in soup.find_all('a', href=True):
            href = tag_a['href']

            # Filtra links de XML, ZIP ou rotas de download
            if any(termo in href.lower() for termo in ['.xml', '.zip', 'download', 'obterarquivo']):
                if href.startswith('/'):
                    href = f"https://pit.tce.pr.gov.br{href}"
                links_arquivos.append(href)

        if not links_arquivos:
            print(f"[-] Nenhum arquivo encontrado para Paranavaí em {ANO}.")
            return

        print(f"[V] Encontrados {len(links_arquivos)} pacotes de arquivos para baixar.")

        for indice, url_download in enumerate(links_arquivos):
            print(f"   -> Baixando pacote {indice + 1}...")

            arquivo_res = session.get(url_download, timeout=45)
            if arquivo_res.status_code == 200:

                # Define um nome temporário para o arquivo compactado
                caminho_zip = os.path.join(PASTA_DESTINO, f"temporario_{indice + 1}.zip")

                # Salva o arquivo ZIP temporariamente no disco
                with open(caminho_zip, 'wb') as f:
                    f.write(arquivo_res.content)

                # --- NOVIDADE: EXTRAÇÃO DO ZIP ---
                try:
                    # Verifica se o arquivo baixado é realmente um ZIP válido
                    if zipfile.is_zipfile(caminho_zip):
                        with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
                            # Extrai todo o conteúdo (os arquivos XML) para a pasta destino
                            zip_ref.extractall(PASTA_DESTINO)
                        print(f"   [V] Pacote {indice + 1} extraído com sucesso em: {PASTA_DESTINO}")
                    else:
                        # Se o site dizia que era zip mas veio um XML direto, apenas renomeia
                        caminho_xml = os.path.join(PASTA_DESTINO, f"dados_{indice + 1}.xml")
                        os.rename(caminho_zip, caminho_xml)
                        print(f"   [V] Arquivo baixado diretamente como XML: {caminho_xml}")

                except Exception as erro_zip:
                    print(f"   [-] Erro ao extrair o ZIP {indice + 1}: {erro_zip}")
                finally:
                    # Remove o arquivo .zip temporário para não acumular lixo no disco
                    if os.path.exists(caminho_zip) and zipfile.is_zipfile(caminho_zip):
                        os.remove(caminho_zip)
            else:
                print(f"   [-] Falha ao baixar o pacote: {url_download}")

            time.sleep(1)

    except Exception as e:
        print(f"[-] Ocorreu um erro durante a automação: {e}")

if __name__ == "__main__":
    baixar_e_extrair_zip()
    print("\n[FIM] Todos os arquivos foram baixados e descompactados!")

[+] Acessando o painel do TCE-PR para: PARANAVAÍ (2026)...
[V] Encontrados 8 pacotes de arquivos para baixar.
   -> Baixando pacote 1...
   [V] Pacote 1 extraído com sucesso em: ./downloads_tce_pr/PARANAVAÍ/2026
   -> Baixando pacote 2...
   [V] Pacote 2 extraído com sucesso em: ./downloads_tce_pr/PARANAVAÍ/2026
   -> Baixando pacote 3...
   [V] Pacote 3 extraído com sucesso em: ./downloads_tce_pr/PARANAVAÍ/2026
   -> Baixando pacote 4...
   [V] Pacote 4 extraído com sucesso em: ./downloads_tce_pr/PARANAVAÍ/2026
   -> Baixando pacote 5...
   [V] Pacote 5 extraído com sucesso em: ./downloads_tce_pr/PARANAVAÍ/2026
   -> Baixando pacote 6...
   [V] Pacote 6 extraído com sucesso em: ./downloads_tce_pr/PARANAVAÍ/2026
   -> Baixando pacote 7...
   [V] Pacote 7 extraído com sucesso em: ./downloads_tce_pr/PARANAVAÍ/2026
   -> Baixando pacote 8...
   [V] Pacote 8 extraído com sucesso em: ./downloads_tce_pr/PARANAVAÍ/2026

[FIM] Todos os arquivos foram baixados e descompactados!
